In [1]:
import random
import math
import pandas as pd
import numpy as np
import time


from preferences import prefs
from user import user_prefs, experiments

startPoint = [59.927085, 30.317504]
endPoint = [59.935408, 30.327108]

In [2]:
df = pd.read_csv('data/places.csv')

In [3]:
POPULATION_SIZE = 100
GENERATIONS = 100

MIN_ROUTE_POINTS = 2
MAX_ROUTE_POINTS = 15

MUTATION_RATE = 0.2
TOURNAMENT_SIZE = 5

def point_interest_score(point_row):
    score = 0
    for category, tags in prefs.items():
        weight = user_prefs[category]
        for tag in tags:
            if tag in point_row.index and point_row[tag]:
                score += weight

    return score

def fitness_vector(route, df):

    distance = route_distance(route, df)
    interest = route_interest_score(route, df)

    diversity = diversity_bonus(route, df)
    chaos = route_smoothness_penalty(route, df)

    return [
        -distance,   # минимизируем расстояние
        interest,    # максимизируем интерес
        diversity,   # максимизируем разнообразие
        -chaos       # минимизируем хаос
    ]

def dominates(f1, f2):
    """
    f1 доминирует f2 если:
    - не хуже по всем метрикам
    - лучше хотя бы по одной
    """

    return (
        all(a >= b for a, b in zip(f1, f2)) and
        any(a > b for a, b in zip(f1, f2))
    )

def fast_non_dominated_sort(population):

    # population = список особей (dict)
    # работаем ТОЛЬКО через индексы

    S = {}      # кого доминирует i
    n = {}      # сколько доминируют i
    rank = {}   # ранг фронта

    fronts = [[]]

    # =====================================================
    # 1. Инициализация
    # =====================================================

    for i in range(len(population)):

        S[i] = []
        n[i] = 0

        for j in range(len(population)):

            if i == j:
                continue

            fi = population[i]["fitness"]
            fj = population[j]["fitness"]

            # i доминирует j
            if dominates(fi, fj):
                S[i].append(j)

            # j доминирует i
            elif dominates(fj, fi):
                n[i] += 1

        if n[i] == 0:
            rank[i] = 0
            fronts[0].append(i)

    # =====================================================
    # 2. Формирование фронтов
    # =====================================================

    current_front = 0

    while len(fronts[current_front]) > 0:

        next_front = []

        for i in fronts[current_front]:

            for j in S[i]:

                n[j] -= 1

                if n[j] == 0:
                    rank[j] = current_front + 1
                    next_front.append(j)

        current_front += 1
        fronts.append(next_front)

    # убираем последний пустой фронт
    fronts = fronts[:-1]

    # =====================================================
    # 3. Преобразуем индексы обратно в особи
    # =====================================================

    sorted_fronts = []

    for front in fronts:
        sorted_fronts.append([population[i] for i in front])

    return sorted_fronts


def crowding_distance(front):

    n = len(front)

    if n == 0:
        return {}

    distance = [0.0 for _ in range(n)]

    num_obj = len(front[0]["fitness"])

    for m in range(num_obj):

        idx = list(range(n))

        idx.sort(key=lambda i: front[i]["fitness"][m])

        distance[idx[0]] = float("inf")
        distance[idx[-1]] = float("inf")

        min_f = front[idx[0]]["fitness"][m]
        max_f = front[idx[-1]]["fitness"][m]

        if max_f == min_f:
            continue

        for i in range(1, n - 1):

            prev_f = front[idx[i - 1]]["fitness"][m]
            next_f = front[idx[i + 1]]["fitness"][m]

            distance[idx[i]] += (
                (next_f - prev_f) / (max_f - min_f)
            )

    return distance


def select_parent(fronts):

    front = random.choice(fronts)

    return random.choice(front)

# Расстояние между точками в метрах
def haversine(lat1, lon1, lat2, lon2):
    R = 6371000

    phi1 = math.radians(lat1)
    phi2 = math.radians(lat2)

    dphi = math.radians(lat2 - lat1)
    dlambda = math.radians(lon2 - lon1)

    a = (
        math.sin(dphi / 2) ** 2
        + math.cos(phi1)
        * math.cos(phi2)
        * math.sin(dlambda / 2) ** 2
    )

    return 2 * R * math.atan2(math.sqrt(a), math.sqrt(1 - a))

# Генерация случайной особи
def create_individual(df):

    route_size = random.randint(
        MIN_ROUTE_POINTS,
        MAX_ROUTE_POINTS
    )

    route = random.sample(
        range(len(df)),
        route_size
    )

    return {
        "route": route,
        "fitness": None
    }

# Создание популяции
def create_population(df):
    return [
        create_individual(df)
        for _ in range(POPULATION_SIZE)
    ]

# Длина маршрута
def route_distance(route, df):

    total_distance = 0

    prev_lat, prev_lon = startPoint

    for idx in route:
        point = df.iloc[idx]

        total_distance += haversine(
            prev_lat,
            prev_lon,
            point["lat"],
            point["lon"]
        )

        prev_lat = point["lat"]
        prev_lon = point["lon"]

    total_distance += haversine(
        prev_lat,
        prev_lon,
        endPoint[0],
        endPoint[1]
    )

    return total_distance

# Кроссовер
def crossover(parent1, parent2):

    route1 = parent1["route"]
    route2 = parent2["route"]

    if len(route1) < 2 or len(route2) < 2:
        return {
            "route": route1.copy(),
            "fitness": None
        }

    cut1 = random.randint(1, len(route1) - 1)
    cut2 = random.randint(1, len(route2) - 1)

    child_route = route1[:cut1]

    for point in route2[cut2:]:
        if point not in child_route:
            child_route.append(point)

    return {
        "route": child_route,
        "fitness": None
    }


# Мутация
def mutate(individual, df):

    route = individual["route"][:]

    if random.random() > MUTATION_RATE:
        return individual

    mutation_type = random.choice([
        "swap",
        "remove",
        "add",
        "shuffle"
    ])

    # -----------------------------------------
    # Swap
    # -----------------------------------------
    if mutation_type == "swap" and len(route) >= 2:

        i, j = random.sample(range(len(route)), 2)
        route[i], route[j] = route[j], route[i]

    # -----------------------------------------
    # Remove
    # -----------------------------------------
    elif mutation_type == "remove":

        if len(route) > MIN_ROUTE_POINTS:
            idx = random.randint(0, len(route) - 1)
            route.pop(idx)

    # -----------------------------------------
    # Add
    # -----------------------------------------
    elif mutation_type == "add":

        if len(route) < MAX_ROUTE_POINTS:

            available = list(
                set(range(len(df))) - set(route)
            )

            if available:

                count_to_add = random.randint(1, 3)

                for _ in range(count_to_add):

                    if not available:
                        break

                    new_point = random.choice(available)
                    available.remove(new_point)

                    insert_pos = random.randint(
                        0,
                        len(route)
                    )

                    route.insert(insert_pos, new_point)

    # -----------------------------------------
    # Shuffle
    # -----------------------------------------
    elif mutation_type == "shuffle":

        random.shuffle(route)

    individual["route"] = route
    individual["fitness"] = None

    return individual

def nsga2(df):

    population = create_population(df)

    for ind in population:
        ind["fitness"] = fitness_vector(ind["route"], df)

    for gen in range(GENERATIONS):

        offspring = []

        while len(offspring) < POPULATION_SIZE:

            # selection (пока можно оставить простым)
            p1 = random.choice(population)
            p2 = random.choice(population)

            child = crossover(p1, p2)
            child = mutate(child, df)

            child["fitness"] = fitness_vector(child["route"], df)

            offspring.append(child)

        combined = population + offspring

        fronts = fast_non_dominated_sort(combined)

        new_population = []

        # =================================================
        # NSGA-II SELECTION STEP (фикс)
        # =================================================

        for front in fronts:

            if len(new_population) + len(front) > POPULATION_SIZE:

                # сортируем по crowding distance
                distances = crowding_distance(front)

                front_sorted = sorted(
                    range(len(front)),
                    key=lambda i: distances[i],
                    reverse=True
                )

                front = [front[i] for i in front_sorted]

                for ind in front:
                    if len(new_population) < POPULATION_SIZE:
                        new_population.append(ind)

                break

            new_population.extend(front)

        population = new_population

        print(f"Gen {gen} done")

    return fronts[0]

In [4]:
# Полный маршрут
def project_point_to_line(point, start, end):
    """
    Проекция точки на линию START -> END.
    Возвращает параметр t:
    0   = начало линии
    1   = конец линии
    >1  = дальше конца
    <0  = до начала
    """

    px, py = point
    sx, sy = start
    ex, ey = end

    line_vec = np.array([ex - sx, ey - sy])
    point_vec = np.array([px - sx, py - sy])

    line_len_sq = np.dot(line_vec, line_vec)

    if line_len_sq == 0:
        return 0

    t = np.dot(point_vec, line_vec) / line_len_sq

    return t

def build_full_route(best_individual, df):

    route_points = []

    # ==========================================
    # Получаем точки маршрута
    # ==========================================

    points = []

    for idx in best_individual["route"]:

        point = df.iloc[idx]

        lat = float(point["lat"])
        lon = float(point["lon"])

        # --------------------------------------
        # Позиция вдоль линии START -> END
        # --------------------------------------

        t = project_point_to_line(
            [lat, lon],
            startPoint,
            endPoint
        )

        points.append({
            "name": point["name"],
            "lat": lat,
            "lon": lon,
            "t": t
        })

    # ==========================================
    # Сортировка по географическому порядку
    # ==========================================

    points.sort(key=lambda x: x["t"])

    # ==========================================
    # Финальный маршрут
    # ==========================================

    route_points.append({
        "name": "START",
        "lat": startPoint[0],
        "lon": startPoint[1]
    })

    route_points.extend(points)

    route_points.append({
        "name": "END",
        "lat": endPoint[0],
        "lon": endPoint[1]
    })

    return route_points

    route_points = []

    route_points.append({
        "name": "START",
        "lat": startPoint[0],
        "lon": startPoint[1]
    })

    for idx in best_individual["route"]:

        point = df.iloc[idx]

        route_points.append({
            "name": point["name"],
            "lat": point["lat"],
            "lon": point["lon"]
        })

    route_points.append({
        "name": "END",
        "lat": endPoint[0],
        "lon": endPoint[1]
    })

    return route_points

# Ссылка на яндекс карты
def generate_yandex_maps_url(route_points):
    """
    route_points:
    [
        {"name": "...", "lat": ..., "lon": ...},
        ...
    ]
    """

    rtext = "~".join(
        f"{point['lat']},{point['lon']}"
        for point in route_points
    )

    yandex_url = (
        f"https://yandex.ru/maps/"
        f"?mode=routes"
        f"&rtext={rtext}"
        f"&rtt=pd"
    )

    return yandex_url

In [5]:
def route_interest_score(route, df):
    total = 0

    for idx in route:
        point = df.iloc[idx]
        total += point_interest_score(point)

    return total


# =========================================================
# Бонус за разнообразие маршрута
# =========================================================

def diversity_bonus(route, df):
    """
    Поощряем разнообразие категорий.
    """

    used_categories = set()

    for idx in route:

        point = df.iloc[idx]

        for category_name, tags in prefs.items():

            for tag in tags:

                if tag in point and point[tag]:
                    used_categories.add(category_name)

    return len(used_categories) * 15


# =========================================================
# Штраф за хаотичность маршрута
# =========================================================

def route_smoothness_penalty(route, df):

    if len(route) < 3:
        return 0

    penalty = 0
    prev_direction = None

    points = []

    points.append(startPoint)

    for idx in route:

        point = df.iloc[idx]

        points.append([
            float(point["lat"]),
            float(point["lon"])
        ])

    points.append(endPoint)

    for i in range(len(points) - 1):

        p1 = points[i]
        p2 = points[i + 1]

        # ВЕКТОР ДВИЖЕНИЯ (lat/lon)
        direction = (
            p2[0] - p1[0],
            p2[1] - p1[1]
        )

        if prev_direction is not None:

            dot = (
                direction[0] * prev_direction[0] +
                direction[1] * prev_direction[1]
            )

            # если резкий разворот
            if dot < 0:
                penalty += 50

        prev_direction = direction

    return penalty

def choose_best_solution(pareto_front, df, user_prefs):

    best = None
    best_score = -1

    for sol in pareto_front:

        route = sol["route"]

        interest = route_interest_score(route, df)
        distance = route_distance(route, df)

        # -----------------------------
        # персонализация
        # -----------------------------

        score = (
            interest * 1.0
            - distance * 0.001
        )

        # можно добавить bias пользователя
        score *= (1 + sum(user_prefs.values()) / 20)

        if score > best_score:
            best_score = score
            best = sol

    return best

def run_experiment(df, user_prefs, experiment_name="exp"):

    start_time = time.time()

    pareto_front = nsga2(df)

    best_individual = choose_best_solution(pareto_front, df, user_prefs)

    route = best_individual["route"]

    final_route = build_full_route(best_individual, df)

    end_time = time.time()

    # -------------------------------
    # метрики
    # -------------------------------

    interest = route_interest_score(route, df)
    distance = route_distance(route, df)
    duration = end_time - start_time

    yandex_url = generate_yandex_maps_url(final_route)

    route_text = route_to_names(final_route)

    return {
        "experiment": experiment_name,
        "interest_score": interest,
        "distance": distance,
        "time_sec": duration,
        "route_length": len(route),
        "route_text": route_text,
        "yandex_url": yandex_url
    }


def run_multiple_experiments(df, experiments):

    results = []

    for exp in experiments:

        print(f"\nRunning: {exp['name']}")

        result = run_experiment(
            df=df,
            user_prefs=exp["user_prefs"],
            experiment_name=exp["name"]
        )

        results.append(result)

        print(f"Done: {exp['name']} | time: {result['time_sec']:.2f}s")

    return results

def route_to_names(final_route):

    names = []

    for point in final_route:

        if point["name"] not in ["START", "END"]:
            names.append(point["name"])

    return " -> ".join(names)

def save_results(results, filename="nsga_experiments.csv"):

    rows = []

    for r in results:

        rows.append({
            "experiment": r["experiment"],
            "interest_score": r["interest_score"],
            "distance": r["distance"],
            "time_sec": r["time_sec"],
            "route_length": r["route_length"],
            "route_text": r["route_text"],
            "yandex_url": r["yandex_url"]
        })

    df_results = pd.DataFrame(rows)
    df_results.to_csv(filename, index=False)

    print(f"\nSaved to {filename}")

results = run_multiple_experiments(df, experiments)

save_results(results)


Running: user_1_military_focus
Gen 0 done
Gen 1 done
Gen 2 done
Gen 3 done
Gen 4 done
Gen 5 done
Gen 6 done
Gen 7 done
Gen 8 done
Gen 9 done
Gen 10 done
Gen 11 done
Gen 12 done
Gen 13 done
Gen 14 done
Gen 15 done
Gen 16 done
Gen 17 done
Gen 18 done
Gen 19 done
Gen 20 done
Gen 21 done
Gen 22 done
Gen 23 done
Gen 24 done
Gen 25 done
Gen 26 done
Gen 27 done
Gen 28 done
Gen 29 done
Gen 30 done
Gen 31 done
Gen 32 done
Gen 33 done
Gen 34 done
Gen 35 done
Gen 36 done
Gen 37 done
Gen 38 done
Gen 39 done
Gen 40 done
Gen 41 done
Gen 42 done
Gen 43 done
Gen 44 done
Gen 45 done
Gen 46 done
Gen 47 done
Gen 48 done
Gen 49 done
Gen 50 done
Gen 51 done
Gen 52 done
Gen 53 done
Gen 54 done
Gen 55 done
Gen 56 done
Gen 57 done
Gen 58 done
Gen 59 done
Gen 60 done
Gen 61 done
Gen 62 done
Gen 63 done
Gen 64 done
Gen 65 done
Gen 66 done
Gen 67 done
Gen 68 done
Gen 69 done
Gen 70 done
Gen 71 done
Gen 72 done
Gen 73 done
Gen 74 done
Gen 75 done
Gen 76 done
Gen 77 done
Gen 78 done
Gen 79 done
Gen 80 done
Gen 81